# Notebook 4: CUDA Building Blocks for LLMs

Follow a small token through operations that appear in a Transformer. Start with a simple Python reference, then run educational kernels on an A100.


## Tensor Map

```text
token_id: scalar
embedding table: [vocab_size, hidden_size]
hidden state: [hidden_size]
attention scores: [sequence_length, sequence_length]
projection weight: [hidden_size, output_size]
logits: [vocab_size]
```

A real model also includes batch, sequence, heads, and head dimension. The shapes stay small here so you can inspect every value.


In [ ]:
from pathlib import Path
import math
import shutil
import subprocess

def find_repo_root(start: Path) -> Path:
    start = start.resolve()
    for candidate in (start, *start.parents):
        if (candidate / "CMakeLists.txt").is_file():
            return candidate
    raise RuntimeError("Run this notebook from inside the cuda-kernels-a100-beginners repository")

ROOT = find_repo_root(Path.cwd())
print("repo:", ROOT)


## Small Reference: Token -> Embedding -> Residual -> RMSNorm

The reference does not replace CUDA. It provides expected behavior for comparison.


In [ ]:
embeddings = [
    [1.0, 0.0, 0.0, 0.0],
    [1.0, 2.0, 3.0, 4.0],
    [0.0, 0.0, 1.0, 0.0],
]
token_id = 1
residual_update = [0.5, 0.5, 0.5, 0.5]
hidden = embeddings[token_id][:]
hidden = [x + update for x, update in zip(hidden, residual_update)]
mean_square = sum(x * x for x in hidden) / len(hidden)
normalized = [x / math.sqrt(mean_square + 1e-5) for x in hidden]
print("token_id:", token_id)
print("embedding shape: [4], value:", embeddings[token_id])
print("after residual shape: [4], value:", hidden)
print("after RMSNorm shape: [4], value:", normalized)
lm_head = [
    [1.0, 0.0, 0.0],
    [0.0, 1.0, 0.0],
    [0.0, 0.0, 1.0],
    [1.0, 1.0, 1.0],
]
expected_logits = [
    sum(normalized[i] * lm_head[i][token] for i in range(len(normalized)))
    for token in range(3)
]
print("expected logits:", expected_logits)
assert len(normalized) == 4


## Causal Mask and Stable Softmax

A query position may see only itself and earlier positions. Softmax subtracts the maximum to prevent overflow.


In [ ]:
sequence_length = 4
mask = [[0.0 if key <= query else -math.inf for key in range(sequence_length)]
        for query in range(sequence_length)]
for row in mask:
    print(row)

# Query 1 may attend only to keys 0 and 1. Add the mask before softmax.
scores = [1000.0, 1001.0, 1002.0, 1003.0]
query = 1
masked_scores = [score + mask[query][key] for key, score in enumerate(scores)]
maximum = max(masked_scores)
exps = [math.exp(score - maximum) for score in masked_scores]
probabilities = [value / sum(exps) for value in exps]
print("masked scores:", masked_scores)
print("softmax:", probabilities, "sum:", sum(probabilities))
assert abs(sum(probabilities) - 1.0) < 1e-12
assert probabilities[2:] == [0.0, 0.0]


## Build and Run the CUDA Examples

The next cell requires an A100, CUDA Toolkit, and CMake. It fails explicitly when a prerequisite is missing.


In [ ]:
required = ("nvidia-smi", "nvcc", "cmake")
missing = [command for command in required if shutil.which(command) is None]
assert not missing, f"Missing required commands: {missing}"
print(subprocess.run(["nvidia-smi", "-L"], text=True, capture_output=True, check=True).stdout)
subprocess.run([
    "cmake", "-S", str(ROOT), "-B", str(ROOT / "build"),
    "-DCMAKE_BUILD_TYPE=Release", "-DCMAKE_CUDA_ARCHITECTURES=80"
], check=True)
subprocess.run(["cmake", "--build", str(ROOT / "build"), "--target", "00_device_query", "-j"], check=True)
device = subprocess.run([str(ROOT / "build/00_device_query")], text=True, capture_output=True, check=True)
print(device.stdout)
device_zero = device.stdout.split("Device 1:", 1)[0]
assert "Device 0:" in device_zero
assert "A100" in device_zero, "CUDA device 0 is not identified as an A100; set CUDA_VISIBLE_DEVICES"
assert "compute capability: 8.0" in device_zero, "CUDA device 0 is not compute capability 8.0"
subprocess.run(["cmake", "--build", str(ROOT / "build"), "-j"], check=True)


In [ ]:
targets = [
    "llm_01_token_embedding",
    "llm_02_residual_add",
    "llm_03_silu_activation",
    "llm_04_rmsnorm",
    "llm_05_causal_mask",
    "llm_06_attention_softmax",
    "llm_07_linear_projection",
    "llm_08_mini_transformer_step",
]
for target in targets:
    result = subprocess.run([str(ROOT / "build" / target)], text=True, capture_output=True)
    print(target, "->", result.stdout.strip())
    if result.stderr:
        print(result.stderr)
    assert result.returncode == 0
    assert "PASS" in result.stdout
    if target == "llm_08_mini_transformer_step":
        logits_line = next(line for line in result.stdout.splitlines() if line.startswith("LOGITS "))
        cuda_logits = [float(value) for value in logits_line.split()[1:]]
        assert len(cuda_logits) == len(expected_logits)
        assert all(abs(actual - expected) < 1e-5
                   for actual, expected in zip(cuda_logits, expected_logits))


## What You Should Be Able to Explain

1. Why is embedding lookup not softmax?
2. Why is residual addition simply vector addition?
3. Where does RMSNorm require reduction and `__syncthreads()`?
4. Why is a causal mask a two-dimensional problem?
5. Why does stable softmax subtract the maximum?
6. Why is a linear projection a matrix-vector multiplication?
7. Which production operations might benefit from a fused kernel?
